In [1]:
from utils_00 import *
# exec(open('c00_utils.py').read())

pn.extension()  # 启用 Jupyter 支持
pd.set_option('expand_frame_repr', False)
pd.set_option('display.max_rows', 1000)
pd.set_option('display.max_colwidth', 100)

In [2]:
native_exchange_data_dir = os.path.join(base_dir, "Native_Exchange_Data")
temporary_experimental_data_dir = os.path.join(base_dir, "Temporary_Experimental_Data")
pro = ts.pro_api(secret_dict['tushare']['api_key'])

def get_exchange_variety_map():
    # 交易所日历路径 NED_dimension_exchange_calendar_path
    NED_dimension_exchange_calendar_path = os.path.join(native_exchange_data_dir, 'dimension_exchange_calendar.csv')
    dimension_exchange_calendar = pd.read_csv(NED_dimension_exchange_calendar_path, dtype = {'date': str})

    # 获取 交易所-品种 表单
    exchange_list = [col for col in dimension_exchange_calendar.columns if col != 'date'] # 交易所列表
    map_list = []
    for exchange in exchange_list:

        exchange_variety_map = pd.DataFrame()
        for attempt in range(3):
            try:
                exchange_variety_map = pro.fut_basic(exchange = exchange, fut_type = "1", fields = ["fut_code"])
                break
            except Exception as e:
                print(f"第 {attempt + 1} 次获取 {exchange} 品种信息失败: {e}")

        exchange_variety_map = exchange_variety_map.drop_duplicates()
        exchange_variety_map["exchange"] = exchange
        map_list.append(exchange_variety_map)

    exchange_variety_map = pd.concat(map_list, ignore_index = True)
    exchange_variety_map = exchange_variety_map.groupby('exchange')['fut_code'].apply(list)
    exchange_variety_map = exchange_variety_map.to_dict()
    return exchange_variety_map

if __name__ == '__main__':
    exchange_variety_map = get_exchange_variety_map()
    exchange_variety_map.pop("CZCE")
    exchange_variety_map.pop("DCE")
    print(exchange_variety_map)

{'CFFEX': ['T', 'TF', 'IF', 'IM', 'IC', 'IH', 'TS', 'TL'], 'GFEX': ['LC', 'SI', 'PS', 'PD', 'PT'], 'INE': ['NR', 'SC', 'LU', 'BC', 'SCTAS', 'EC'], 'SHFE': ['AO', 'NI', 'AL', 'HC', 'BU', 'RU', 'FU', 'CU', 'AG', 'ZN', 'AU', 'PB', 'SS', 'RB', 'SN', 'WR', 'SP', 'BR', 'AD', 'OP']}


In [3]:
exchange = None
variety = None

if __name__ == '__main__':
    # 交易所选择组件
    exchange_radio = widgets.RadioButtons(
        options = list(exchange_variety_map.keys()),
        description = '交易所选择 exchange:',
        layout = {'width': '400px'}
    )

    # 品种选择组件（初始禁用）
    species_toggle = widgets.ToggleButtons(
        options = [],
        description = '品种选择 variety:',
        disabled = True,
        button_style = '',
        style = {'button_width': 'auto'},
        layout = widgets.Layout(width = '600px'),
    )

    # 输出提示区域
    out = widgets.Textarea(
        value = '请先选择交易所，然后选择品种',
        layout = {'width': '400px', 'height': '50px'},
        disabled = True
    )

    display(widgets.VBox([exchange_radio, species_toggle, out]))

    def on_exchange_change(change):
        """当交易所改变时，更新可选品种"""
        global exchange
        if change['name'] == 'value':
            exchange = change['new']
            varieties = exchange_variety_map.get(exchange, [])
            species_toggle.options = sorted(varieties)
            species_toggle.disabled = len(varieties) == 0
            if varieties:
                species_toggle.value = varieties[0]  # 默认选第一个
                out.value = f'已选择交易所：{exchange}，请选择品种'
            else:
                out.value = f'交易所 {exchange} 无可用品种'

    def on_species_change(change):
        """当品种改变时，记录到全局变量"""
        global variety
        if change['name'] == 'value' and species_toggle.options:
            variety = change['new']
            out.value = f'已选择交易所 exchange: {exchange}\n已选择品种 variety: {variety}'

    # 绑定事件 ["SCTAS", "SI", "FU", "AU", "SS", "WR"]
    exchange_radio.observe(on_exchange_change, names = 'value')
    species_toggle.observe(on_species_change, names = 'value')

In [4]:
# 品种文件夹路径 NED_variety_path
NED_variety_path = os.path.join(str(native_exchange_data_dir), str(variety))

# 合约日历路径 NED_dimension_symbol_calendar_path
NED_dimension_symbol_calendar_path = os.path.join(NED_variety_path, 'dimension_symbol_calendar.csv')
dimension_symbol_calendar = pd.read_csv(NED_dimension_symbol_calendar_path, dtype = {'date': str})

symbol_daily_timeseries_dict = dict()
for column in dimension_symbol_calendar.columns:
    if column == 'date':  continue
    else:  symbol = column

    # 品种文件夹路径 NED_variety_path
    NED_variety_path = os.path.join(str(native_exchange_data_dir), str(variety))

    # 时间序列文件夹路径 NED_variety_timeseries_path
    NED_variety_timeseries_path = os.path.join(NED_variety_path , "timeseries_data")

    # 日线行情文件夹路径 NED_futures_daily_timeseries_path
    NED_futures_daily_timeseries_path = os.path.join(NED_variety_timeseries_path, "futures_daily_timeseries")

    # 合约日线行情路径 NED_fact_futures_daily_timeseries_path
    NED_fact_futures_daily_timeseries_path = os.path.join(NED_futures_daily_timeseries_path, f'{symbol}.csv')
    symbol_daily_timeseries_dict[symbol] = pd.read_csv(NED_fact_futures_daily_timeseries_path, dtype = {'date': str})


# 打印 symbol_daily_timeseries_dict 数据结构
print(len(symbol_daily_timeseries_dict), symbol_daily_timeseries_dict.keys(), end = '\n\n')
for symbol, symbol_timeseries in symbol_daily_timeseries_dict.items():
    print(symbol)
    print(symbol_timeseries.info())
    print(symbol_timeseries.head(2))

197 dict_keys(['RB1101', 'RB1102', 'RB1103', 'RB1104', 'RB1105', 'RB1106', 'RB1107', 'RB1108', 'RB1109', 'RB1110', 'RB1111', 'RB1112', 'RB1201', 'RB1202', 'RB1203', 'RB1204', 'RB1205', 'RB1206', 'RB1207', 'RB1208', 'RB1209', 'RB1210', 'RB1211', 'RB1212', 'RB1301', 'RB1302', 'RB1303', 'RB1304', 'RB1305', 'RB1306', 'RB1307', 'RB1308', 'RB1309', 'RB1310', 'RB1311', 'RB1312', 'RB1401', 'RB1402', 'RB1403', 'RB1404', 'RB1405', 'RB1406', 'RB1407', 'RB1408', 'RB1409', 'RB1410', 'RB1411', 'RB1412', 'RB1501', 'RB1502', 'RB1503', 'RB1504', 'RB1505', 'RB1506', 'RB1507', 'RB1508', 'RB1509', 'RB1510', 'RB1511', 'RB1512', 'RB1601', 'RB1602', 'RB1603', 'RB1604', 'RB1605', 'RB1606', 'RB1607', 'RB1608', 'RB1609', 'RB1610', 'RB1611', 'RB1612', 'RB1701', 'RB1702', 'RB1703', 'RB1704', 'RB1705', 'RB1706', 'RB1707', 'RB1708', 'RB1709', 'RB1710', 'RB1711', 'RB1712', 'RB1801', 'RB1802', 'RB1803', 'RB1804', 'RB1805', 'RB1806', 'RB1807', 'RB1808', 'RB1809', 'RB1810', 'RB1811', 'RB1812', 'RB1901', 'RB1902', 'RB19

#### 01. 主力判定

In [5]:
date_to_main_contract_dict = {}  # {date: dict_keys(['symbol_list', 'current_main', 'main_volume', 'secondary_contract', 'nearest_contract', 'main_switch'])}
current_main = None

data_values = dimension_symbol_calendar.iloc[:, 1:] # 跳过第一列 date 提取除之外的所有数值列
date_range = dimension_symbol_calendar.loc[
    (data_values == 1).any(axis = 1)  # 检查每一行是否有至少一个 合约在线，以筛选出交易日列表
]['date'].tolist()

# 计算每个日期的在线合约数量，当满足数量等于 12 时开始拼接
# 合约日历前 11 个月没有完整的过往 12 个月份上线的合约，故需要 main_continuous_start_date 截断不完整的部分。完成截断后，剩余交易日进行近月合约交割与新最远月合约上线间隙，偶尔只有 11 个合约在线
symbol_list = dimension_symbol_calendar.columns.difference(['date']) # 合约列表
row_sums = np.sum(
    dimension_symbol_calendar[symbol_list].values, # 合约日历中 dimension_symbol_calendar 横向求和，得到每行用 1 标记为有交易的合约的数量
    axis = 1 )
main_continuous_start_date = dimension_symbol_calendar.loc[np.where(row_sums >= 12)[0][0], 'date'] # 主力合约拼接起始日期 str 格式
# row_sums >= 12 → [False, False, True, True, False, True]
# np.where(...) → (array([2, 3, 5]),)
# np.where(row_sums >= 12)[0][0] 取出 np.array, 再取出首个索引值

# 数据错乱修正
date_to_main_contract_dict_correction = {
    'HC': '20150101',   # 2014 年末多个合约在数日间交替被判定为主力
    'AL': '20150526',  # 2015.05 前存在大量交替判定为主力的合约
}
if variety in date_to_main_contract_dict_correction:
    main_continuous_start_date = max( # 数据错乱，丢弃部分多个合约在一段时间内交替成为主力合约的部分。
        date_to_main_contract_dict_correction[variety],
        main_continuous_start_date
    ) # 取较大值一次性进行截取
date_range = sorted([date for date in date_range if date >= main_continuous_start_date])




for index, date in enumerate(date_range):
    current_date = datetime.datetime.strptime(date, '%Y%m%d')
    daily_volumes = {} # 当天各合约交易量

    date_to_main_contract_dict[date] = dict()

    row = dimension_symbol_calendar.loc[dimension_symbol_calendar['date'] == date]
    date_to_main_contract_dict[date].update({
        'symbol_list': row.columns[row.iloc[0] == 1].tolist()
    })

    for symbol, symbol_timeseries_daily in symbol_daily_timeseries_dict.items():
        day_data = symbol_timeseries_daily[symbol_timeseries_daily['date'] == date]

        if not day_data.empty:
            daily_volumes[symbol] = day_data['vol'].iloc[0] # 若这一天 date 在合约中有值
        else:
            daily_volumes[symbol] = 0

    if not daily_volumes or sum(daily_volumes.values()) == 0:
        if not index > 0:
            date_to_main_contract_dict[date].update({  # 对于第一个日期，初始化所有键为 None/0
                'current_main': current_main,
                'main_volume': 0,
                'secondary_contract': None,
                'nearest_contract': None,
                'main_switch': False
            })
            continue

        prev_date = date_range[index - 1]
        if prev_date in date_to_main_contract_dict:  # 使用 get 方法安全获取，避免 KeyError
            prev_data = date_to_main_contract_dict[prev_date]

            # current_main = prev_data.get('current_main') # 使用 current_main 变量，而不是从前一天复制
            main_volume = 0  # 当天无交易量，所以设为0
            secondary_contract = prev_data.get('secondary_contract')
            nearest_contract = prev_data.get('nearest_contract')

            date_to_main_contract_dict[date].update({
                'current_main': current_main,
                'main_volume': main_volume,
                'secondary_contract': secondary_contract,
                'nearest_contract': nearest_contract,
                'main_switch': False  # 无交易量时不切换主力
            })

        continue

    # 确定本日主力合约
    if current_main is None:
        current_main = max(daily_volumes, key = lambda symbol: daily_volumes.get(symbol)) # 第一天选择交易量最大的作为主力

    # 确定近月合约
    nearest_contract = None
    current_year = current_date.year
    current_month = current_date.month

    current_year_month = current_year * 12 + current_month # 计算当前年份和月份的数值表示
    min_month_diff = float('inf') # 取无限大作为首次最小值比较的对象

    for symbol in daily_volumes.keys(): # 遍历所有合约，找到月份代码最接近当前月份的合约
        if daily_volumes[symbol] == 0: # 跳过没有交易量的合约
            continue

        try: # 提取年份和月份

            year_code = int('20' + symbol[-4: -2]) # 假设合约代码格式为: 品种代码 + 年份(2位) + 月份(2位)，如 RB2501 表示 2025 年 1 月
            month_code = int(symbol[-2:]) # 月份

            contract_year_month = year_code * 12 + month_code # 计算合约年份月份的数值表示
            month_diff = contract_year_month - current_year_month # 计算与当前月份的差值

            if month_diff >= 0 and month_diff < min_month_diff: # 只考虑未来到期的合约，包括当月
                min_month_diff = month_diff
                nearest_contract = symbol

        except (ValueError, IndexError):
            continue # 如果合约代码格式不符合预期，跳过


    # 确定副主力合约
    secondary_contract = None
    secondary_volume = 0

    for symbol, volume in daily_volumes.items():
        if symbol == current_main: # 排除主力合约
            continue

        if volume > secondary_volume:
            secondary_volume = volume
            secondary_contract = symbol


    # 记录当前日期的主力合约信息
    main_volume = daily_volumes.get(current_main, 0)
    date_to_main_contract_dict[date].update({
        'current_main': current_main,
        'main_volume': main_volume,
        'secondary_contract': secondary_contract,
        'nearest_contract': nearest_contract,
    })

    # 判定下一天是否主力切换
    potential_new = None
    main_switch = False
    for symbol, volume in daily_volumes.items():
        if not symbol != current_main: continue
        if not volume > main_volume * 1.1: continue
        if potential_new is None or volume > daily_volumes.get(potential_new, 0): # 大于旧主力 1.1 倍的合约不一定只有一个
            potential_new = symbol

    if potential_new:
        date_idx = date_range.index(date)
        if date_idx + 1 < len(date_range): # 处理主力合约切换，完成当日主力合约标记后，更新 current_main 变量
            current_main = potential_new # 更新变量后，遍历到下一个日期时将根据此变量进行计算
            main_switch = True

    date_to_main_contract_dict[date].update({
        'main_switch': main_switch,
        'current_main': current_main,
    })

    print(f"日期: {date}, 主力: {current_main}, 副主力: {secondary_contract}, 近月: {nearest_contract}, 主力切换: {main_switch}")


# 创建主力合约日期映射字典
main_contract_to_date_list_dict = {}

for date, contract_data in date_to_main_contract_dict.items():
    main_contract = contract_data['current_main']
    main_contract_to_date_list_dict[main_contract] = main_contract_to_date_list_dict.get(main_contract, []) # 使用 get 方法
    main_contract_to_date_list_dict[main_contract].append(date) # 将日期添加到对应合约的列表中

for contract in main_contract_to_date_list_dict: # 对每个合约的日期列表进行排序
    main_contract_to_date_list_dict[contract].sort()

print(
    '\n\n'
    f'main_contract_to_date_list_dict 长度验证: {sum([len(_) for _ in main_contract_to_date_list_dict.values()])}  \n' # main_contract_to_date_list_dict 覆盖日期的长度验证
    f'date_to_main_contract_dict 长度验证: {len(date_to_main_contract_dict)}'
)

日期: 20101216, 主力: RB1105, 副主力: RB1110, 近月: RB1101, 主力切换: False
日期: 20101217, 主力: RB1105, 副主力: RB1110, 近月: RB1101, 主力切换: False
日期: 20101220, 主力: RB1105, 副主力: RB1110, 近月: RB1101, 主力切换: False
日期: 20101221, 主力: RB1105, 副主力: RB1110, 近月: RB1101, 主力切换: False
日期: 20101222, 主力: RB1105, 副主力: RB1110, 近月: RB1101, 主力切换: False
日期: 20101223, 主力: RB1105, 副主力: RB1110, 近月: RB1101, 主力切换: False
日期: 20101224, 主力: RB1105, 副主力: RB1110, 近月: RB1101, 主力切换: False
日期: 20101227, 主力: RB1105, 副主力: RB1110, 近月: RB1101, 主力切换: False
日期: 20101228, 主力: RB1105, 副主力: RB1110, 近月: RB1101, 主力切换: False
日期: 20101229, 主力: RB1105, 副主力: RB1110, 近月: RB1101, 主力切换: False
日期: 20101230, 主力: RB1105, 副主力: RB1110, 近月: RB1101, 主力切换: False
日期: 20101231, 主力: RB1105, 副主力: RB1110, 近月: RB1101, 主力切换: False
日期: 20110104, 主力: RB1105, 副主力: RB1110, 近月: RB1101, 主力切换: False
日期: 20110105, 主力: RB1105, 副主力: RB1110, 近月: RB1101, 主力切换: False
日期: 20110106, 主力: RB1105, 副主力: RB1110, 近月: RB1101, 主力切换: False
日期: 20110107, 主力: RB1105, 副主力: RB1110, 近月: RB1101, 主力切换

#### 02. 主力片段拼接

In [6]:
dataset_start_boundary_dt = None   # 时间起始点 datetime 对象
dataset_end_boundary_dt = None     # 时间结束点 datetime 对象
dataset_start_boundary = ""        # 时间起始点字符串
dataset_end_boundary = ""          # 时间结束点字符串

if __name__ == '__main__':

    # 使用主力判断环节的 date_range 提取 start_date_dt
    start_date_dt = datetime.datetime.strptime(date_range[0], "%Y%m%d")
    end_date_dt = datetime.datetime.now()  # 获取当前时间
    end_date_yesterday_dt = end_date_dt - datetime.timedelta(days = 1)  # 昨天作为最晚日期

    # 生成起始边界月份按钮
    start_month_buttons = []  # 创建起始月份按钮容器
    from dateutil.relativedelta import relativedelta  # 计算最早日期前 6 个月开始的时间
    for i in range(1, 7, 1):  # 生成 6 个按钮
        month_date = start_date_dt + relativedelta(months = (i))
        month_str = month_date.strftime("%Y-%m")
        btn = widgets.Button(
            description = month_str,
            layout = {'width': '85px', 'height': '25px'},
            button_style = ''
        )
        start_month_buttons.append(btn)
    start_month_button_row = widgets.HBox(start_month_buttons)


    # 生成结束边界月份按钮
    end_month_buttons = []  # 创建结束月份按钮容器
    from dateutil.relativedelta import relativedelta # 计算最后日期倒数 6 个月开始的时间
    for i in range(5, -1, -1):  # 生成 6 个按钮
        month_date = end_date_dt - relativedelta(months = i)
        month_str = month_date.strftime("%Y-%m")
        btn = widgets.Button(
            description = month_str,
            layout = {'width': '85px', 'height': '25px'},
            button_style = ''
        )
        end_month_buttons.append(btn)
    end_month_button_row = widgets.HBox(end_month_buttons)

    # 创建自定义起始时间输入框
    dataset_start_boundary_input = widgets.Text(
        value = '',
        placeholder = '格式: 2025-01-01',
        description = 'dataset_start_boundary:',
        layout = {'width': '300px', 'height': '25px'},
        style = {'description_width': '130px'},
    )

    # 创建自定义结束时间输入框
    dataset_end_boundary_input = widgets.Text(
        value = '',
        placeholder = '格式: 2025-01-01',
        description = 'dataset_end_boundary:',
        layout = {'width': '300px', 'height': '25px'},
        style = {'description_width': '130px'},
    )

    # 输出显示区域
    display_output = widgets.Textarea(
        value = '起始时间 dataset_start_boundary: 未选择\n结束时间 dataset_end_boundary: 未选择',
        layout = {'width': '530px', 'height': '47px'},
        disabled = True
    )

    # 显示所有组件
    display(widgets.VBox([ # 垂直布局盒子 Vertical Box
        widgets.Label('起始边界 dataset_start_boundary:'),
        dataset_start_boundary_input,
        start_month_button_row,
        widgets.Label('结束边界 dataset_end_boundary:'),
        dataset_end_boundary_input,
        end_month_button_row,
        display_output
    ]))

    def update_display():
        """更新显示区域"""
        global dataset_start_boundary, dataset_end_boundary
        display_output.value = f'起始时间 dataset_start_boundary: {dataset_start_boundary}\n结束时间 dataset_end_boundary: {dataset_end_boundary}'

    def set_start_time_from_month(month_str):
        """根据月份设置起始时间为该月初 21:01:00"""
        global dataset_start_boundary_dt, dataset_start_boundary
        year, month = map(int, month_str.split('-'))  # 解析年月
        dataset_start_boundary_dt = datetime.datetime(year, month, 1, 9, 1, 0)
        dataset_start_boundary = dataset_start_boundary_dt.strftime("%Y-%m-%d 09:01:00")

        # 更新自定义输入框显示
        dataset_start_boundary_input.value = dataset_start_boundary_dt.strftime("%Y-%m-%d")
        update_display()

    def set_end_time_from_month(month_str):
        """根据月份设置结束时间为该月末 21:01:00"""
        global dataset_end_boundary_dt, dataset_end_boundary
        year, month = map(int, month_str.split('-'))  # 解析年月
        if month == 12:  # 计算下个月的第一天，然后减去 1 秒得到本月最后一秒
            next_month = datetime.datetime(year + 1, 1, 1)
        else:
            next_month = datetime.datetime(year, month + 1, 1)

        dataset_end_boundary_dt = next_month - datetime.timedelta(seconds = 1)
        dataset_end_boundary = dataset_end_boundary_dt.strftime("%Y-%m-%d 21:01:00")

        # 如果选择的结束时间晚于昨天，则调整为昨天
        if dataset_end_boundary_dt > end_date_yesterday_dt:
            dataset_end_boundary_dt = datetime.datetime(
                end_date_yesterday_dt.year, end_date_yesterday_dt.month, end_date_yesterday_dt.day,
                21, 1, 0
            )
            dataset_end_boundary = dataset_end_boundary_dt.strftime("%Y-%m-%d 21:01:00")

        # 更新自定义输入框显示
        dataset_end_boundary_input.value = dataset_end_boundary_dt.strftime("%Y-%m-%d")
        update_display()

    def set_dataset_start_boundary_time(date_str):
        """设置自定义起始日期（格式：2025-01-01）"""
        global dataset_start_boundary_dt, dataset_start_boundary

        cleaned_str = date_str.strip()  # 清理输入值 date_str 的首尾空格
        year, month, day = map(int, cleaned_str.split('-')) # 解析日期
        dataset_start_boundary_dt = datetime.datetime(year, month, day, 9, 1, 0)
        dataset_start_boundary = dataset_start_boundary_dt.strftime("%Y-%m-%d 09:01:00")

        update_display()

    def set_dataset_end_boundary_time(date_str):
        """设置自定义结束日期（格式：2025-01-01）"""
        global dataset_end_boundary_dt, dataset_end_boundary

        cleaned_str = date_str.strip()  # 清理输入值 date_str 的首尾空格
        year, month, day = map(int, cleaned_str.split('-')) # 解析日期
        dataset_end_boundary_dt = datetime.datetime(year, month, day, 21, 1, 0)
        dataset_end_boundary = dataset_end_boundary_dt.strftime("%Y-%m-%d 21:01:00")

        # 如果自定义的结束时间晚于昨天，则调整为昨天
        if dataset_end_boundary_dt > end_date_yesterday_dt:
            dataset_end_boundary_dt = datetime.datetime(
                end_date_yesterday_dt.year, end_date_yesterday_dt.month, end_date_yesterday_dt.day,
                21, 1, 0
            )
            dataset_end_boundary = dataset_end_boundary_dt.strftime("%Y-%m-%d 21:01:00")

        update_display()

    # 事件处理函数
    def on_start_month_button_click(btn):
        """起始月份按钮 点击事件"""
        set_start_time_from_month(btn.description)

    def on_end_month_button_click(btn):
        """结束月份按钮 点击事件"""
        set_end_time_from_month(btn.description)

    def on_dataset_start_boundary_input_change(change):
        """自定义起始时间输入改变事件"""
        if change['name'] == 'value':
            date_str = change['new']
            if date_str: # 非空则更新
                set_dataset_start_boundary_time(date_str)

    def on_dataset_end_boundary_input_change(change):
        """自定义结束时间输入改变事件"""
        if change['name'] == 'value':
            date_str = change['new']
            if date_str: # 非空则更新
                set_dataset_end_boundary_time(date_str)

    # 绑定事件
    for btn in start_month_buttons:
        btn.on_click(on_start_month_button_click)

    for btn in end_month_buttons:
        btn.on_click(on_end_month_button_click)

    dataset_start_boundary_input.observe(on_dataset_start_boundary_input_change, names = 'value')
    dataset_end_boundary_input.observe(on_dataset_end_boundary_input_change, names = 'value')

    # 初始化：默认选择起始边界最早日期前6个月中的第一个，结束边界最近一个月
    first_start_btn = start_month_buttons[0]
    last_end_btn = end_month_buttons[-1]
    set_start_time_from_month(first_start_btn.description)
    set_end_time_from_month(last_end_btn.description)
    update_display()

In [7]:
# 品种文件夹路径 NED_variety_path
NED_variety_path = os.path.join(str(native_exchange_data_dir), str(variety))

# 时间序列文件夹路径 NED_variety_timeseries_path
NED_variety_timeseries_path = os.path.join(NED_variety_path , "timeseries_data")

# 分钟行情时间序列文件夹路径 NED_futures_minutely_timeseries_path
NED_futures_minutely_timeseries_path = os.path.join(NED_variety_timeseries_path, "futures_minutely_timeseries")

# 使用主力判断环节的 date_range 提取 start_date_dt
main_continuous_start_date = date_range[0]

# 主力连续合约分钟行情拼接
main_continuous_OHLC_timeseries = pd.DataFrame({'date': []})  # 初始化 main_continuous_OHLC_timeseries 变量
main_continuous_Volumn_timeseries = pd.DataFrame({'date': []})  # 初始化 main_continuous_Volumn_timeseries 变量
main_continuous_Money_timeseries = pd.DataFrame({'date': []})  # 初始化 main_continuous_Money_timeseries 变量
main_continuous_OpenInterest_timeseries = pd.DataFrame({'date': []})  # 初始化 main_continuous_OpenInterest_timeseries 变量

main_segment_timepoint_list_dict = {}  # 记录每个主力片段起始与结束时间点
close_of_previous_main_segment = None  # 记录前一个主力合约片段结束时的收盘价
roll_over_gap_list = []  # 合约衔接处价差列表
for symbol in symbol_list:

    # 分钟行情时间序列文件路径 NED_fact_futures_minutely_timeseries_path
    NED_fact_futures_minutely_timeseries_path = os.path.join(NED_futures_minutely_timeseries_path, f'{symbol}.csv')
    fact_futures_minutely_timeseries = pd.read_csv(NED_fact_futures_minutely_timeseries_path, dtype = {'date': str, })

    fact_futures_minutely_timeseries = fact_futures_minutely_timeseries.loc[
        fact_futures_minutely_timeseries['date'] >=
        str(datetime.datetime.strptime(main_continuous_start_date, '%Y%m%d').replace(hour = 9, minute = 1, second = 0))  # 用主力连续起始的早盘开盘点截取 fact_futures_minutely_timeseries
    ].reset_index(drop = True).copy()  # 重置索引并将旧索引丢弃，以方便后续根据索引是否为 0 判断合约主力片段起始点

    fact_futures_minutely_timeseries = fact_futures_minutely_timeseries.loc[
        (fact_futures_minutely_timeseries['date'] >= dataset_start_boundary) &  # 使用 数据集起始边界 dataset_start_boundary 与 结束边界 dataset_end_boundary
        (fact_futures_minutely_timeseries['date'] < dataset_end_boundary)  # 切割指定时间范围的行情数据
    ].reset_index(drop = True).copy()  # 重置索引并将旧索引丢弃，以方便后续根据索引是否为 0 判断合约主力片段起始点


    # 本次循环合约过程变量 DataFrame
    symbol_OHLC_timeseries = fact_futures_minutely_timeseries[['date', 'open', 'high', 'low', 'close']]
    symbol_Volumn_timeseries = fact_futures_minutely_timeseries[['date', 'volume']]
    symbol_Money_timeseries = fact_futures_minutely_timeseries[['date', 'money']]
    symbol_OpenInterest_timeseries = fact_futures_minutely_timeseries[['date', 'open_interest']]

    date_list = main_contract_to_date_list_dict.get(symbol, []) # 只有少数品种每个月份的合约都存在主力片段，大部分品种仅有季度月合约或半年度月合约，即 (3, 6, 9, 12) 滚动为主力
    date_list = [date for date in date_list if date >= main_continuous_start_date] # 获取的主力合约片段日期列表需截取 main_contract_start_date 之后的日期，前期数据失真
    date_list = [
        date for date in date_list if (date >= datetime.datetime.strptime(dataset_start_boundary, "%Y-%m-%d %H:%M:%S").strftime("%Y%m%d")) and
        (date <= datetime.datetime.strptime(dataset_end_boundary, "%Y-%m-%d %H:%M:%S").strftime("%Y%m%d"))
    ] # 使用 数据集起始边界 dataset_start_boundary 与 结束边界 dataset_end_boundary，切割指定时间范围的行情数据


    if date_list: # 若存在主力时间段，则拼接处理其 OHLC 数据

        main_segment_start_date = date_list[0] # 主力合约起始日期
        main_segment_end_date = date_list[- 1] # 主力合约结束日期

        start_date_morning_session_time_point = datetime.datetime.strptime(main_segment_start_date, '%Y%m%d').replace(hour = 9, minute = 1, second = 0)
        start_date_morning_session_time_point = start_date_morning_session_time_point.strftime('%Y-%m-%d %H:%M:%S')  # 主力合约首日早盘起始时间点


        # print(symbol_OHLC_timeseries['date']) # 时间判定逻辑错误定位
        # print(date_list)
        print(symbol_OHLC_timeseries[symbol_OHLC_timeseries['date'] == start_date_morning_session_time_point])
        start_date_morning_session_time_pointindex = symbol_OHLC_timeseries[symbol_OHLC_timeseries['date'] == start_date_morning_session_time_point].index[0]
        if start_date_morning_session_time_pointindex == 0: # 如果本次循环的主力合约交易首日早盘起始时间点对应的索引为 0 ，即为数据的第一行，
            # 则该时间点是合约主力片段的起始时间点
            main_segment_start_time_point = start_date_morning_session_time_point

        else: # 否则需考虑上一个交易日有无夜盘
            previous_trading_end_point = symbol_OHLC_timeseries.at[start_date_morning_session_time_pointindex - 1, 'date']
            if not (
                (previous_trading_end_point.split()[1] > '21:00:00') or # 判断上一个交易日期最后一个盘面的收盘点是否落在 21:00 到 09:00 之间
                (previous_trading_end_point.split()[1] < '09:00:00') # 部分交易日的夜盘没有持续到凌晨，而是结束在 23:00。故这个语句能严谨检查上一个交易日是否有夜盘
            ): # 上一行数据若为午盘收盘，则 end_point 为 15:15 或 15:00，若为夜盘则不同品种有不同时间点
                # 虽然不同品种夜盘收盘点不同，但可以肯定所有收盘点都发生在 09:00 早盘开盘前，所有可以由于上一行数据是否位于 09:00 早盘开盘前到 前一交易日的 21:00 后来断定上一个交易日有无夜盘交易
                main_segment_start_time_point = start_date_morning_session_time_point # 上一个交易日无夜盘交易，则无需考虑将上一个夜盘合并到本次循环的主力合约片段中

            else:
                # 否则上一个交易日的夜盘应视为本合约主力片段的一部分，计算 main_segment_start_time_point 为合约主力片段起始点
                # 需注意 previous_trading_end_point 是从分钟行情数据提取的，格式为 %Y-%m-%d %H:%M:%S
                main_segment_start_time_point = (
                    datetime.datetime
                    .strptime(previous_trading_end_point, '%Y-%m-%d %H:%M:%S')
                    .replace(hour = 21, minute = 1, second = 0)
                )
                if (
                    (previous_trading_end_point.split()[1] > '00:00:00') and
                    (previous_trading_end_point.split()[1] < '09:00:00')
                ): # 若夜盘结束于次日凌晨，则日期取值需要减 1
                    main_segment_start_time_point = main_segment_start_time_point - datetime.timedelta(days = 1) # 获取前一个夜盘的结束时间点的日期，夜盘结束于该日期凌晨，开始于该前一个日期的 21:01

                main_segment_start_time_point = main_segment_start_time_point.strftime('%Y-%m-%d %H:%M:%S') # 主力合约首日夜盘起始时间点

        print(symbol)
        print(start_date_morning_session_time_point, )
        print(main_segment_start_time_point, end = '\n\n')

        for hour, minute in [(15, 15), (15, 0)]: # 少部分品种午盘 15:15 结束, 大部分品种午盘 15:00 结束
            main_segment_end_time_point = (
                datetime.datetime.strptime(main_segment_end_date, '%Y%m%d')
                .replace(hour = hour, minute = minute, second = 0)
                .strftime('%Y-%m-%d %H:%M:%S')
            )
            if main_segment_end_time_point in symbol_OHLC_timeseries['date']: # 先尝试 15:15 是否存在于 OHLC 数据的时间列中, 若不存在，则判断该品种午盘结束时间为 15:00
                break # 遍历两个选项 [(15, 15), (15, 0)] 满足时间点存在则中断 for loop

        # 从 OHLC 时间序列原始数据切割出主力片段
        main_segment_OHLC_timeseries = symbol_OHLC_timeseries[
            (symbol_OHLC_timeseries['date'] >= main_segment_start_time_point) &
            (symbol_OHLC_timeseries['date'] <= main_segment_end_time_point)
        ]

        # 记录主力片段的起始与结束时间点
        main_segment_timepoint_list_dict[symbol] = {}
        main_segment_timepoint_list_dict[symbol]['main_segment_start_time_point'] = main_segment_start_time_point # 主力片段起始时间点
        main_segment_timepoint_list_dict[symbol]['main_segment_end_time_point'] = main_segment_end_time_point # 主力片段结束时间点

        # 计算主力切换缺口 roll_over_gap，使用缺口进行前复权操作，并将缺口记录进 roll_over_gap_list
        if not close_of_previous_main_segment is None: # 如果 close_of_previous_main_segment 无记录，说明首次发生主力拼接
            roll_over_gap = main_segment_OHLC_timeseries['open'].iloc[0] - close_of_previous_main_segment # 否则根据 close_of_previous_main_segment 所记录的上一个合约的收盘价
            roll_over_gap_list.append(roll_over_gap) # 得到主力切换缺口 roll_over_gap

            main_continuous_OHLC_timeseries[['open', 'high', 'low', 'close']] += roll_over_gap # 用 roll_over_gap 前复权已采集的 main_segment_OHLC_timeseries

        close_of_previous_main_segment = main_segment_OHLC_timeseries['close'].iloc[-1] # 完成切换缺口计算，或跳过缺口计算后
        # 将 close_of_previous_main_segment 更新为本次主力片段结束点的收盘价

        main_continuous_OHLC_timeseries = pd.concat(
            [main_continuous_OHLC_timeseries, main_segment_OHLC_timeseries], ignore_index = True
        ).copy()





    # 交易量加总合并 main_continuous_Volumn_timeseries
    if main_continuous_Volumn_timeseries.empty:
        main_continuous_Volumn_timeseries = symbol_Volumn_timeseries
    else:
        main_continuous_Volumn_timeseries = pd.merge(
            main_continuous_Volumn_timeseries,
            symbol_Volumn_timeseries,
            on = 'date',
            how = 'outer',
            suffixes = ('_1', '_2')
        )
        main_continuous_Volumn_timeseries['volume'] = (
            main_continuous_Volumn_timeseries['volume_1'].fillna(0) +
            main_continuous_Volumn_timeseries['volume_2'].fillna(0)
        )
        main_continuous_Volumn_timeseries = main_continuous_Volumn_timeseries[['date', 'volume']].sort_values('date')

    # 交易额加总合并 main_continuous_Money_timeseries
    if main_continuous_Money_timeseries.empty:
        main_continuous_Money_timeseries = symbol_Money_timeseries
    else:
        main_continuous_Money_timeseries = pd.merge(
            main_continuous_Money_timeseries,
            symbol_Money_timeseries,
            on = 'date',
            how = 'outer',
            suffixes = ('_1', '_2')
        )
        main_continuous_Money_timeseries['money'] = (
            main_continuous_Money_timeseries['money_1'].fillna(0) +
            main_continuous_Money_timeseries['money_2'].fillna(0)
        )
        main_continuous_Money_timeseries = main_continuous_Money_timeseries[['date', 'money']].sort_values('date')

    # 持仓量加总合并 main_continuous_OpenInterest_timeseries
    if main_continuous_OpenInterest_timeseries.empty:
        main_continuous_OpenInterest_timeseries = symbol_OpenInterest_timeseries
    else:
        main_continuous_OpenInterest_timeseries = pd.merge(
            main_continuous_OpenInterest_timeseries,
            symbol_OpenInterest_timeseries,
            on = 'date',
            how = 'outer',
            suffixes = ('_1', '_2')
        )
        main_continuous_OpenInterest_timeseries['open_interest'] = (
            main_continuous_OpenInterest_timeseries['open_interest_1'].fillna(0) +
            main_continuous_OpenInterest_timeseries['open_interest_2'].fillna(0)
        )
        main_continuous_OpenInterest_timeseries = main_continuous_OpenInterest_timeseries[['date', 'open_interest']].sort_values('date')


# 主力复权价差滚动分配 main_continuous_OHLC_timeseries
roll_over_gap_sum = sum(roll_over_gap_list) # 所有主力片段拼接点的总价差
last_main_symbol = sorted(list(main_segment_timepoint_list_dict.keys()))[-1] # 最后一个主力片段合约代码
mask = main_continuous_OHLC_timeseries['date'] < main_segment_timepoint_list_dict[last_main_symbol]['main_segment_start_time_point'] # 提取出最后一个主力片段的起始时间点，
# 计算 mark 掩码，用于提取最后一个主力片段前的主力连续数据，保证最后一个主力片段不参与返复权，即最后一个主力片段（当前正在交易的主力片段）原始价格序列不被扭曲

decreasing_array = np.linspace(roll_over_gap_sum, 0, mask.sum())
OHLC_columns = ['open', 'high', 'low', 'close']
for column in OHLC_columns:
    main_continuous_OHLC_timeseries.loc[mask, column] = (
        main_continuous_OHLC_timeseries
        .loc[mask, column]
        .values - decreasing_array
    )

# 将所有列拼接成完整的 分钟轴行情时间序列 main_continuous_minutely_timeseries
main_continuous_minutely_timeseries = pd.concat([
    main_continuous_OHLC_timeseries.set_index('date', drop = True),
    main_continuous_Volumn_timeseries.set_index('date', drop = True),
    main_continuous_Money_timeseries.set_index('date', drop = True),
    main_continuous_OpenInterest_timeseries.set_index('date', drop = True),
], axis = 1)

main_continuous_minutely_timeseries = main_continuous_minutely_timeseries.reset_index()

                  date    open    high     low   close
0  2015-01-05 09:01:00  2550.0  2556.0  2546.0  2554.0
RB1505
2015-01-05 09:01:00
2015-01-05 09:01:00

                      date    open    high     low   close
16500  2015-03-03 09:01:00  2581.0  2587.0  2580.0  2585.0
RB1510
2015-03-03 09:01:00
2015-03-02 21:01:00

                      date    open    high     low   close
54615  2015-07-14 09:01:00  2083.0  2084.0  2076.0  2079.0
RB1601
2015-07-14 09:01:00
2015-07-13 21:01:00

                      date    open    high     low   close
51825  2015-11-03 09:01:00  1786.0  1788.0  1785.0  1787.0
RB1605
2015-11-03 09:01:00
2015-11-02 21:01:00

                      date    open    high     low   close
45795  2016-03-11 09:01:00  2059.0  2068.0  2059.0  2068.0
RB1610
2016-03-11 09:01:00
2016-03-10 21:01:00

                      date    open    high     low   close
57015  2016-08-17 09:01:00  2580.0  2587.0  2580.0  2586.0
RB1701
2016-08-17 09:01:00
2016-08-16 21:01:00

            

In [13]:
import pandas as pd
import numpy as np
import os
import datetime

print("=" * 80)
print("【数据断档假说验证诊断】")
print("=" * 80)

# ==============================================================================
# 1. CU2605 原始 CSV 在切换窗口的逐小时数据分布
# ==============================================================================
print("\n" + "=" * 80)
print("1. CU2605 原始 CSV 在 2026-03-17 ~ 2026-03-20 的逐小时数据分布")
print("=" * 80)

if 'NED_futures_minutely_timeseries_path' in globals():
    fp = globals()['NED_futures_minutely_timeseries_path']
    csv_2605 = os.path.join(fp, "CU2605.csv")

    if os.path.exists(csv_2605):
        df_raw = pd.read_csv(csv_2605, dtype={'date': str})
        print(f"  原始数据总行数: {len(df_raw)}")
        print(f"  全局范围: {df_raw['date'].iloc[0]} ~ {df_raw['date'].iloc[-1]}")

        # 复现原代码的两层截取
        df = df_raw.loc[df_raw['date'] >= '2010-12-16 09:01:00'].copy()
        df = df.loc[(df['date'] >= '2011-01-01 09:01:00') & (df['date'] < '2026-04-30 21:01:00')].copy().reset_index(drop=True)
        print(f"\n  截取后总行数: {len(df)}")
        print(f"  截取后范围: {df['date'].iloc[0]} ~ {df['date'].iloc[-1]}")

        # 逐日逐时段统计
        target_dates = ['2026-03-17', '2026-03-18', '2026-03-19', '2026-03-20']
        for d in target_dates:
            sub = df[df['date'].str.startswith(d)]
            print(f"\n  [{d}] 截取后数据: {len(sub)} 行")
            if len(sub) > 0:
                times = sub['date'].str.split().str[1]
                print(f"    首条: {sub['date'].iloc[0]}")
                print(f"    末条: {sub['date'].iloc[-1]}")
                print(f"    时段分布:")
                # 按小时分组统计
                sub['hour'] = times.str[:2]
                hour_counts = sub.groupby('hour').size().sort_index()
                for h, c in hour_counts.items():
                    print(f"      {h}:00~  {c} 行")
    else:
        print("  [ERROR] CU2605.csv 不存在")
else:
    print("  [SKIP] NED_futures_minutely_timeseries_path 未定义")

# ==============================================================================
# 2. 复现 CU2605 的 start_time_point 计算，精确定位前一行
# ==============================================================================
print("\n" + "=" * 80)
print("2. 复现 CU2605 main_segment_start_time_point 计算过程")
print("=" * 80)

if 'NED_futures_minutely_timeseries_path' in globals():
    # 重新读取并截取，确保状态干净
    fp = globals()['NED_futures_minutely_timeseries_path']
    csv_2605 = os.path.join(fp, "CU2605.csv")
    df = pd.read_csv(csv_2605, dtype={'date': str})
    df = df.loc[df['date'] >= '2010-12-16 09:01:00'].copy()
    df = df.loc[(df['date'] >= '2011-01-01 09:01:00') & (df['date'] < '2026-04-30 21:01:00')].copy().reset_index(drop=True)

    # 构造 OHLC 子集（模拟 symbol_OHLC_timeseries）
    ohlc = df[['date', 'open', 'high', 'low', 'close']].copy()

    morning = '2026-03-19 09:01:00'
    matches = ohlc[ohlc['date'] == morning]

    if len(matches) == 0:
        print(f"  [ERROR] {morning} 不在截取后的数据中")
    else:
        idx = matches.index[0]
        print(f"  目标时间点: {morning}")
        print(f"  在截取后数据中的索引: {idx}")
        print(f"  该索引前后5行数据:")
        context = ohlc.iloc[max(0, idx-3):min(len(ohlc), idx+4)]
        print(context.to_string())

        prev = ohlc.at[idx - 1, 'date']
        prev_dt = datetime.datetime.strptime(prev, '%Y-%m-%d %H:%M:%S')
        curr_dt = datetime.datetime.strptime(morning, '%Y-%m-%d %H:%M:%S')
        delta = curr_dt - prev_dt

        print(f"\n  前一行: {prev}")
        print(f"  当前行: {morning}")
        print(f"  时间差: {delta} ({delta.total_seconds()/3600:.1f} 小时)")

        # 判断时间差是否合理
        if delta <= datetime.timedelta(hours=12):
            print(f"  [OK] 时间差 <= 12 小时，属于正常隔夜/跨周末范围")
        elif delta <= datetime.timedelta(hours=72):
            print(f"  [WARNING] 时间差在 12~72 小时之间，疑似跨周末")
            # 进一步判断是否是周五→周一
            prev_weekday = prev_dt.weekday()
            curr_weekday = curr_dt.weekday()
            print(f"  前一行星期: {prev_weekday} (0=周一), 当前行星期: {curr_weekday}")
            if prev_weekday >= 4 and curr_weekday <= 0:
                print(f"  [CONFIRMED] 跨周末场景，回退逻辑合理")
            else:
                print(f"  [ALERT] 非跨周末但出现大间隔，疑似数据断档！")
        else:
            print(f"  [CRITICAL] 时间差 > 72 小时，严重异常")

        # 复现原代码的 start 计算
        prev_time = prev.split()[1]
        print(f"\n  --- 复现原代码逻辑 ---")
        print(f"  prev_time: {prev_time}")
        cond_overnight = (prev_time > '21:00:00') or (prev_time < '09:00:00')
        print(f"  有夜盘? ({prev_time}>21:00 or <09:00): {cond_overnight}")

        if cond_overnight:
            start_pt = datetime.datetime.strptime(prev, '%Y-%m-%d %H:%M:%S').replace(hour=21, minute=1, second=0)
            print(f"  replace(21:01): {start_pt.strftime('%Y-%m-%d %H:%M:%S')}")

            cond_rollback = (prev_time > '00:00:00') and (prev_time < '09:00:00')
            print(f"  减1天判断? ({prev_time}>00:00 and <09:00): {cond_rollback}")
            if cond_rollback:
                start_pt -= datetime.timedelta(days=1)
                print(f"  减1天后: {start_pt.strftime('%Y-%m-%d %H:%M:%S')}")

                # 关键：判断减1天后是否合理
                expected_earliest = curr_dt - datetime.timedelta(days=1)
                expected_earliest = expected_earliest.replace(hour=21, minute=1, second=0)
                print(f"  当前交易日允许最早起点: {expected_earliest.strftime('%Y-%m-%d %H:%M:%S')}")
                if start_pt < expected_earliest:
                    print(f"  [ROOT CAUSE] 计算出的起点 ({start_pt.strftime('%Y-%m-%d %H:%M:%S')}) 早于允许的最早起点！")
                    print(f"               这说明前一行不是当前交易日的前一夜盘，而是更早的数据。")
                    print(f"               多回退了1天，导致与旧合约片段重叠。")

# ==============================================================================
# 3. 对比：正常切换点 CU2603→CU2604 的同样计算
# ==============================================================================
print("\n" + "=" * 80)
print("3. 对比：正常切换点 CU2604 的 start_time_point 计算")
print("=" * 80)

if 'NED_futures_minutely_timeseries_path' in globals():
    fp = globals()['NED_futures_minutely_timeseries_path']
    csv_2604 = os.path.join(fp, "CU2604.csv")

    if os.path.exists(csv_2604):
        df = pd.read_csv(csv_2604, dtype={'date': str})
        df = df.loc[df['date'] >= '2010-12-16 09:01:00'].copy()
        df = df.loc[(df['date'] >= '2011-01-01 09:01:00') & (df['date'] < '2026-04-30 21:01:00')].copy().reset_index(drop=True)
        ohlc = df[['date', 'open', 'high', 'low', 'close']].copy()

        # CU2604 date_list[0] = 20260225
        morning = '2026-02-25 09:01:00'
        matches = ohlc[ohlc['date'] == morning]

        if len(matches) > 0:
            idx = matches.index[0]
            prev = ohlc.at[idx - 1, 'date']
            prev_dt = datetime.datetime.strptime(prev, '%Y-%m-%d %H:%M:%S')
            curr_dt = datetime.datetime.strptime(morning, '%Y-%m-%d %H:%M:%S')
            delta = curr_dt - prev_dt

            print(f"  目标: {morning}")
            print(f"  前一行: {prev}")
            print(f"  时间差: {delta} ({delta.total_seconds()/3600:.1f} 小时)")

            prev_time = prev.split()[1]
            if (prev_time > '21:00:00') or (prev_time < '09:00:00'):
                start_pt = datetime.datetime.strptime(prev, '%Y-%m-%d %H:%M:%S').replace(hour=21, minute=1, second=0)
                if (prev_time > '00:00:00') and (prev_time < '09:00:00'):
                    start_pt -= datetime.timedelta(days=1)
                print(f"  代码计算起点: {start_pt.strftime('%Y-%m-%d %H:%M:%S')}")

                expected = curr_dt - datetime.timedelta(days=1)
                expected = expected.replace(hour=21, minute=1, second=0)
                print(f"  允许最早起点: {expected.strftime('%Y-%m-%d %H:%M:%S')}")
                print(f"  是否合理: {start_pt >= expected}")

# ==============================================================================
# 4. 检查 CU2604 在重叠区间的数据分布
# ==============================================================================
print("\n" + "=" * 80)
print("4. CU2604 同期数据分布（验证重叠区间）")
print("=" * 80)

if 'NED_futures_minutely_timeseries_path' in globals():
    fp = globals()['NED_futures_minutely_timeseries_path']
    csv_2604 = os.path.join(fp, "CU2604.csv")

    if os.path.exists(csv_2604):
        df = pd.read_csv(csv_2604, dtype={'date': str})
        df = df.loc[df['date'] >= '2010-12-16 09:01:00'].copy()
        df = df.loc[(df['date'] >= '2011-01-01 09:01:00') & (df['date'] < '2026-04-30 21:01:00')].copy().reset_index(drop=True)

        for d in ['2026-03-16', '2026-03-17', '2026-03-18']:
            sub = df[df['date'].str.startswith(d)]
            if len(sub) > 0:
                print(f"\n  [{d}] {len(sub)} 行")
                print(f"    首条: {sub['date'].iloc[0]}")
                print(f"    末条: {sub['date'].iloc[-1]}")
            else:
                print(f"\n  [{d}] 无数据")

        # 检查 CU2604 的 date_list[-1] = 20260318 对应的片段
        # 按设计应包含 2026-03-17 夜盘 ~ 2026-03-18 日盘
        overlap_start = '2026-03-17 21:01:00'
        overlap_end = '2026-03-18 01:00:00'
        sub_overlap = df[(df['date'] >= overlap_start) & (df['date'] <= overlap_end)]
        print(f"\n  CU2604 在 [{overlap_start} ~ {overlap_end}] 区间有 {len(sub_overlap)} 行数据")
        if len(sub_overlap) > 0:
            print(f"    这意味着如果 CU2605 也从 2026-03-17 21:01 开始，两者将重叠 {len(sub_overlap)} 行")

# ==============================================================================
# 5. 检查 CU2605 在 2026-03-18 的数据缺失细节
# ==============================================================================
print("\n" + "=" * 80)
print("5. CU2605 在 2026-03-18 的数据缺失细节")
print("=" * 80)

if 'NED_futures_minutely_timeseries_path' in globals():
    fp = globals()['NED_futures_minutely_timeseries_path']
    csv_2605 = os.path.join(fp, "CU2605.csv")
    df_raw = pd.read_csv(csv_2605, dtype={'date': str})

    # 不截取，看原始文件在 2026-03-18 的全貌
    sub = df_raw[df_raw['date'].str.startswith('2026-03-18')]
    print(f"  原始文件中 2026-03-18 数据: {len(sub)} 行")
    if len(sub) > 0:
        print(f"    首条: {sub['date'].iloc[0]}")
        print(f"    末条: {sub['date'].iloc[-1]}")

        # 检查是否有 09:01~15:00 和 21:01~23:59
        has_day = ((sub['date'].str.split().str[1] >= '09:01:00') & (sub['date'].str.split().str[1] <= '15:15:00')).any()
        has_night = (sub['date'].str.split().str[1] >= '21:01:00').any()
        print(f"    包含日盘(09:01~15:15): {has_day}")
        print(f"    包含夜盘(>=21:01): {has_night}")

        if not has_day and not has_night:
            print(f"  [CONFIRMED] 2026-03-18 完全没有日盘和夜盘，只有凌晨数据！")
            print(f"              这是数据断档，不是正常的跨周末。")

print("\n" + "=" * 80)
print("诊断结束。请根据上述输出确认数据断档假说。")
print("=" * 80)

【数据断档假说验证诊断】

1. CU2605 原始 CSV 在 2026-03-17 ~ 2026-03-20 的逐小时数据分布
  原始数据总行数: 103620
  全局范围: 2025-05-15 21:01:00 ~ 2026-04-24 15:00:00

  截取后总行数: 103620
  截取后范围: 2025-05-15 21:01:00 ~ 2026-04-24 15:00:00

  [2026-03-17] 截取后数据: 465 行
    首条: 2026-03-17 00:00:00
    末条: 2026-03-17 23:59:00
    时段分布:
      00:00~  60 行
      01:00~  1 行
      09:00~  59 行
      10:00~  45 行
      11:00~  31 行
      13:00~  29 行
      14:00~  60 行
      15:00~  1 行
      21:00~  59 行
      22:00~  60 行
      23:00~  60 行

  [2026-03-18] 截取后数据: 61 行
    首条: 2026-03-18 00:00:00
    末条: 2026-03-18 01:00:00
    时段分布:
      00:00~  60 行
      01:00~  1 行

  [2026-03-19] 截取后数据: 404 行
    首条: 2026-03-19 09:01:00
    末条: 2026-03-19 23:59:00
    时段分布:
      09:00~  59 行
      10:00~  45 行
      11:00~  31 行
      13:00~  29 行
      14:00~  60 行
      15:00~  1 行
      21:00~  59 行
      22:00~  60 行
      23:00~  60 行

  [2026-03-20] 截取后数据: 286 行
    首条: 2026-03-20 00:00:00
    末条: 2026-03-20 15:00:00
    时段分布:
    

In [8]:
# 检查 OHLC 序列
ohlc_duplicates = main_continuous_OHLC_timeseries['date'].duplicated().sum()
print(f"main_continuous_OHLC_timeseries 重复日期数量: {ohlc_duplicates}")
if ohlc_duplicates > 0:
    print("重复的日期值:")
    print(main_continuous_OHLC_timeseries[main_continuous_OHLC_timeseries['date'].duplicated(keep = False)]['date'])

# 检查 Volume 序列
volume_duplicates = main_continuous_Volumn_timeseries['date'].duplicated().sum()
print(f"main_continuous_Volumn_timeseries 重复日期数量: {volume_duplicates}")

# 检查 Money 序列
money_duplicates = main_continuous_Money_timeseries['date'].duplicated().sum()
print(f"main_continuous_Money_timeseries 重复日期数量: {money_duplicates}")

# 检查 OpenInterest 序列
oi_duplicates = main_continuous_OpenInterest_timeseries['date'].duplicated().sum()
print(f"main_continuous_OpenInterest_timeseries 重复日期数量: {oi_duplicates}")

# 数据集范围命名字符串拼接 dataset_range_boundary_file_name
if 'main_continuous_minutely_timeseries' in locals():

    dataset_range_boundary_0 = main_continuous_minutely_timeseries.loc[0, 'date']
    dataset_range_boundary_0 = dataset_range_boundary_0.split()[0]

    dataset_range_boundary_1 = main_continuous_minutely_timeseries.tail(1)['date'].iloc[0]
    dataset_range_boundary_1 = dataset_range_boundary_1.split()[0]

    dataset_range_boundary_file_name = f'dataset_range_boundary_{dataset_range_boundary_0}_{dataset_range_boundary_1}'
    print(f'\ndataset_range_boundary_file_name: {dataset_range_boundary_file_name}')

import gc; gc.collect()

main_continuous_OHLC_timeseries 重复日期数量: 0
main_continuous_Volumn_timeseries 重复日期数量: 0
main_continuous_Money_timeseries 重复日期数量: 0
main_continuous_OpenInterest_timeseries 重复日期数量: 0

dataset_range_boundary_file_name: dataset_range_boundary_2015-01-05_2026-03-31


321

In [9]:
# 临时实验数据品种文件夹路径 TED_variety_path
TED_variety_path = os.path.join(str(temporary_experimental_data_dir), str(variety))
os.makedirs(TED_variety_path, exist_ok = True) # 创建必要目录，若不存在

# 时间范围文件夹路径 dataset_range_boundary_path
dataset_range_boundary_path = os.path.join(TED_variety_path, dataset_range_boundary_file_name)
os.makedirs(dataset_range_boundary_path, exist_ok = True) # 创建必要目录，若不存在

if variety == 'RB':  # 螺纹钢 数据异常值修正
    main_continuous_minutely_timeseries.loc[
        main_continuous_minutely_timeseries['date'].isin([
            '2019-09-16 10:15:00', '2019-09-16 10:31:00', '2019-09-16 11:30:00'
        ]), 'volume'
    ] /= 100
    main_continuous_minutely_timeseries.loc[
        main_continuous_minutely_timeseries['date'].isin([
            '2019-09-16 10:15:00', '2019-09-16 10:31:00', '2019-09-16 11:30:00'
        ]), 'money'
    ] /= 100

# 主力片段拼接分钟时间序列文件路径 main_continuous_minutely_timeseries_path
main_continuous_minutely_timeseries_path = os.path.join(dataset_range_boundary_path, f'main_continuous_minutely_timeseries.csv')
main_continuous_minutely_timeseries.to_csv(main_continuous_minutely_timeseries_path, index = False)

#### 3. 主力位置分钟数据拼接

In [10]:
date_range = sorted(list(date_to_main_contract_dict.keys())) # 交易日列表
minutely_timestamp_position_mapping_list = []  # 分钟时间戳位置映射列表 minutely_timestamp_position_mapping_list

# 将主力片段时间点字典转换为 datetime 对象以便时间比较
main_segment_timepoint_datetime_dict = {}  # 主力片段时间点日期时间字典 main_segment_timepoint_datetime_dict
for contract_symbol, time_segment_dict in main_segment_timepoint_list_dict.items():
    segment_start_datetime = datetime.datetime.strptime(
        time_segment_dict['main_segment_start_time_point'],
        '%Y-%m-%d %H:%M:%S'
    )  # 片段开始日期时间 segment_start_datetime
    segment_end_datetime = datetime.datetime.strptime(
        time_segment_dict['main_segment_end_time_point'],
        '%Y-%m-%d %H:%M:%S'
    )  # 片段结束日期时间 segment_end_datetime
    main_segment_timepoint_datetime_dict[contract_symbol] = {
        'main_segment_start_datetime': segment_start_datetime,
        'main_segment_end_datetime': segment_end_datetime
    }



# 遍历每个分钟时间戳
# 初始化三个时间段的时间逻辑处理变量
current_trading_date_str = None
next_trading_date_index = None
next_trading_date_str = None
previous_date_str = None
previous_date_next_trading_date_index= None
previous_date_next_trading_date_str = None

for minutely_timestamp_str in main_continuous_minutely_timeseries['date']:  # 主力连续分钟时间戳序列
    minutely_timestamp_datetime = datetime.datetime.strptime(
        minutely_timestamp_str, '%Y-%m-%d %H:%M:%S')  # 分钟时间戳日期时间 minutely_timestamp_datetime

    current_trading_date_str = minutely_timestamp_datetime.strftime('%Y%m%d') # 今日年月日日期字符串

    previous_date_str = (minutely_timestamp_datetime - datetime.timedelta(days = 1)).strftime('%Y%m%d') # 前一日年月日日期字符串，由于仅在 0:00 到 9:00 间的分钟行涉及前一日
    if previous_date_str in date_range: # 故操作中涉及前一日变量时，前一日一定是交易日
        previous_date_next_trading_date_index= date_range.index(previous_date_str) # 使用前一日在date_range 中的索引，获取前一日的下一交易日，即发生在前一日但归属于前一日下一交易日的夜盘的所属日期
        if previous_date_next_trading_date_index <= (len(date_range) - 1):
            previous_date_next_trading_date_str = date_range[previous_date_next_trading_date_index + 1]
        else:
            previous_date_next_trading_date_index = previous_date_next_trading_date_str = None
    else: previous_date_next_trading_date_index = previous_date_next_trading_date_str = None

    if current_trading_date_str in date_range:
        next_trading_date_index = date_range.index(current_trading_date_str) + 1 # 下一交易日在 date_range 的索引
        if next_trading_date_index <= (len(date_range) - 1):
            next_trading_date_str = date_range[next_trading_date_index] # 当索引不超过 date_raneg 时，获取下一交易日年月日日期字符串
        else:
            next_trading_date_index = next_trading_date_str = None
    else: next_trading_date_index = next_trading_date_str = None

    # main_continuous_minutely_timeseries 在数据集边界 dataset_start_boundary 切割阶段做了严格的时间边界处理
    # 其起点以 9:01 为左边界，即其所有行不小于 date_range 首日的 9：01，不会出现所需判断的行对应的 previous_trading_date_str 超过 date_range 左边界的状况
    # 其终点以 21:00 为右边界，即考虑了多种午盘收盘的时间点 (15:00 & 15:15 等)，即其所有行不会大于 date_range 最后一个交易日的 21:00

    # 确定交易日日期，夜盘属于次日
    # 夜盘时间（21:00及以后）属于下一个交易日，须在 date_range 以索引定位到下一个值
    if minutely_timestamp_datetime.time() >= datetime.time(21, 0) and minutely_timestamp_datetime.time() <= datetime.time(0, 0):
        # 如果分钟行在 21:00 到 0:00 范围内，则归属于今天的下一交易日
        trading_date_str = next_trading_date_str  # 只取年月日即可，用于变换为 str 形式以索引 date_to_main_contract_dict

    if minutely_timestamp_datetime.time() >= datetime.time(21, 0) and minutely_timestamp_datetime.time() <= datetime.time(0, 0):
        # 如果分钟行在 0:00 到 9:00 范围内，则说明昨天为交易日，且该日夜间有夜盘，该夜盘的 0:00 到 9:00 归属于昨天的下一交易日
        # 次日凌晨所延续的，从今日开始的夜盘，属于下一交易日，而下一交易日不一定是次日，故不能直接简单用时间天数加一
        trading_date_str = previous_date_next_trading_date_str

    if minutely_timestamp_datetime.time() >= datetime.time(9, 0) and minutely_timestamp_datetime.time() <= datetime.time(21, 0):
        trading_date_str = current_trading_date_str




    # 初始化位置标记行（12个位置全为0）
    contract_position_marker_row = [0] * 12  # 合约位置标记行 contract_position_marker_row

    # 首先，根据main_segment_timepoint_list_dict确定当前时间点的主力合约
    current_main_contract_symbol = None  # 当前主力合约符号 current_main_contract_symbol
    for contract_symbol, segment_datetime_dict in main_segment_timepoint_datetime_dict.items():
        segment_start = segment_datetime_dict['main_segment_start_datetime']
        segment_end = segment_datetime_dict['main_segment_end_datetime']

        if segment_start <= minutely_timestamp_datetime <= segment_end:
            current_main_contract_symbol = contract_symbol
            break

    # 如果找到主力合约，并且该交易日有数据，则确定位置
    if current_main_contract_symbol and trading_date_str in date_to_main_contract_dict:
        trading_day_contract_info = date_to_main_contract_dict[trading_date_str]  # 交易日合约信息 trading_day_contract_info
        symbol_position_list = trading_day_contract_info['symbol_list']  # 符号位置列表 symbol_position_list

        # 找到主力合约在symbol_list中的位置
        if current_main_contract_symbol in symbol_position_list:
            main_contract_position_index = symbol_position_list.index(current_main_contract_symbol)  # 主力合约位置索引 main_contract_position_index

            # 将对应位置标记为1
            if main_contract_position_index < 12:  # 确保索引在0-11范围内
                contract_position_marker_row[main_contract_position_index] = 1

    # 添加到映射列表
    minutely_timestamp_position_mapping_list.append(
        [minutely_timestamp_str] + contract_position_marker_row
    )

# 创建主力合约分钟位置分布表格
curve_structure_position_distribution_table = pd.DataFrame(
    minutely_timestamp_position_mapping_list,
    columns = ['date'] + [str(i + 1) for i in range(12)]  # 列名为 1~12，代表位置索引
)  # 主力合约分钟位置分布表格 curve_structure_position_distribution_table

print(curve_structure_position_distribution_table.head(10))
print("\n...")
print(curve_structure_position_distribution_table.tail(10))

                  date  1  2  3  4  5  6  7  8  9  10  11  12
0  2015-01-05 09:01:00  0  0  0  0  1  0  0  0  0   0   0   0
1  2015-01-05 09:02:00  0  0  0  0  1  0  0  0  0   0   0   0
2  2015-01-05 09:03:00  0  0  0  0  1  0  0  0  0   0   0   0
3  2015-01-05 09:04:00  0  0  0  0  1  0  0  0  0   0   0   0
4  2015-01-05 09:05:00  0  0  0  0  1  0  0  0  0   0   0   0
5  2015-01-05 09:06:00  0  0  0  0  1  0  0  0  0   0   0   0
6  2015-01-05 09:07:00  0  0  0  0  1  0  0  0  0   0   0   0
7  2015-01-05 09:08:00  0  0  0  0  1  0  0  0  0   0   0   0
8  2015-01-05 09:09:00  0  0  0  0  1  0  0  0  0   0   0   0
9  2015-01-05 09:10:00  0  0  0  0  1  0  0  0  0   0   0   0

...
                       date  1  2  3  4  5  6  7  8  9  10  11  12
962240  2026-03-31 14:51:00  0  1  0  0  0  0  0  0  0   0   0   0
962241  2026-03-31 14:52:00  0  1  0  0  0  0  0  0  0   0   0   0
962242  2026-03-31 14:53:00  0  1  0  0  0  0  0  0  0   0   0   0
962243  2026-03-31 14:54:00  0  1  0  0  0  0

In [11]:
# 临时实验数据品种文件夹路径 TED_variety_path
TED_variety_path = os.path.join(str(temporary_experimental_data_dir), str(variety))

# 时间范围文件夹路径 dataset_range_boundary_path
dataset_range_boundary_path = os.path.join(TED_variety_path, dataset_range_boundary_file_name)

# 主力合约分钟位置分布表格路径 curve_structure_position_distribution_table
curve_structure_position_distribution_table_path = os.path.join(dataset_range_boundary_path, f'curve_structure_position_distribution_table.csv')
curve_structure_position_distribution_table.to_csv(curve_structure_position_distribution_table_path, index = False)

#### 04. 曲率结构时间点拼接

In [12]:
TED_variety_path = os.path.join(str(temporary_experimental_data_dir), str(variety))  # 临时实验数据品种文件夹路径 TED_variety_path
dataset_range_boundary_path = os.path.join(TED_variety_path, dataset_range_boundary_file_name) # 时间范围文件夹路径 dataset_range_boundary_path
main_continuous_minutely_timeseries_path = os.path.join(dataset_range_boundary_path, f'main_continuous_minutely_timeseries.csv')  # 主力片段拼接分钟时间序列文件路径 main_continuous_minutely_timeseries_path
main_continuous_minutely_timeseries = pd.read_csv(main_continuous_minutely_timeseries_path, dtype = {'date': str, })

symbol_to_delistdate = pro.fut_basic(exchange = exchange, fut_type = '1', fields = 'symbol,delist_date').set_index('symbol')
symbol_to_delistdate = symbol_to_delistdate['delist_date'].to_dict()

curve_structure_time_point = main_continuous_minutely_timeseries[['date']].copy()
curve_structure_time_point_raw2D_list = []
for index, row in curve_structure_time_point.iterrows():

    time_point = row['date']
    time_point = datetime.datetime.strptime(time_point, '%Y-%m-%d %H:%M:%S')

    is_in_day_session = datetime.time(9, 0) <= time_point.time() <= datetime.time(15, 0)
    if is_in_day_session:

        curve_structure_symbol_list = date_to_main_contract_dict[
            time_point.strftime('%Y%m%d')
        ]['symbol_list']

        curve_structure_time_point_list = [symbol_to_delistdate[symbol] for symbol in curve_structure_symbol_list]
        curve_structure_time_point_list = [(
            datetime.datetime.strptime(time_point, '%Y%m%d')
            .replace(hour = 15, minute = 0, second = 0)
            .strftime('%Y-%m-%d %H:%M:%S')
        ) for time_point in curve_structure_time_point_list]

    curve_structure_time_point_raw2D_list.append(
        [row['date']] + curve_structure_time_point_list
    )

curve_structure_time_point = pd.DataFrame(
    curve_structure_time_point_raw2D_list,
    columns = ['date'] + [1 + i for i in range(12)]
)

In [13]:
# 临时实验数据品种文件夹路径 TED_variety_path
TED_variety_path = os.path.join(str(temporary_experimental_data_dir), str(variety))

# 时间范围文件夹路径 dataset_range_boundary_path
dataset_range_boundary_path = os.path.join(TED_variety_path, dataset_range_boundary_file_name)

# 曲率结构时间点数据路径 curve_structure_time_point_path
curve_structure_time_point_path = os.path.join(dataset_range_boundary_path, f'curve_structure_time_point.csv')
curve_structure_time_point.to_csv(curve_structure_time_point_path, index = False)

#### 05. 曲率结构分钟数据拼接

In [14]:
def process_curve_structure_timeseries(column = 'open'): # 曲率结构分钟数据提取

    NED_variety_path = os.path.join(str(native_exchange_data_dir), str(variety))  # 品种文件夹路径 NED_variety_path
    NED_variety_timeseries_path = os.path.join(NED_variety_path , "timeseries_data")  # 时间序列文件夹路径 NED_variety_timeseries_path
    NED_futures_minutely_timeseries_path = os.path.join(NED_variety_timeseries_path, "futures_minutely_timeseries")  # 分钟行情时间序列文件夹路径 NED_futures_minutely_timeseries_path

    # 计算每个日期的在线合约数量，当满足数量等于 12 时开始拼接
    symbol_list = dimension_symbol_calendar.columns.difference(['date']) # 合约列表
    row_sums = np.sum(
        dimension_symbol_calendar[symbol_list].values, # 合约日历中 dimension_symbol_calendar 横向求和，得到每行用 1 标记为有交易的合约的数量
        axis = 1
    )
    main_continuous_start_date = dimension_symbol_calendar.loc[np.where(row_sums >= 12)[0][0], 'date'] # 主力合约拼接起始日期

    # 期限结构分钟行情
    curve_structure_timeseries = pd.DataFrame({'date': []})  # 初始化 curve_structure_timeseries 变量
    symbol_list = sorted(symbol_list)  # 把合约以 str 形式排序
    for symbol in symbol_list:
        print(symbol)
        print(curve_structure_timeseries.tail(2), end = '\n\n')

        # 分钟行情时间序列文件路径 NED_fact_futures_minutely_timeseries_path
        NED_fact_futures_minutely_timeseries_path = os.path.join(NED_futures_minutely_timeseries_path, f'{symbol}.csv')
        fact_futures_minutely_timeseries = pd.read_csv(NED_fact_futures_minutely_timeseries_path, dtype = {'date': str, })
        fact_futures_minutely_timeseries = fact_futures_minutely_timeseries[
            fact_futures_minutely_timeseries['date'] >=
            str(datetime.datetime.strptime(main_continuous_start_date, '%Y%m%d').replace( hour = 9, minute = 1, second = 0 ))  # 用主力连续起始的早盘开盘点截取 fact_futures_minutely_timeseries
        ].reset_index(drop = True).copy()  # 重置索引并将旧索引丢弃，以方便后续根据索引是否为 0 判断合约主力片段起始点

        if fact_futures_minutely_timeseries.empty: continue

        # 期限结构分钟数据 curve_structure_timeseries
        current_data = fact_futures_minutely_timeseries[['date', column]].rename(  # 获取当前合约的数据
            columns = {column: symbol}
        )

        # 如果是第一个有效合约，直接初始化
        if curve_structure_timeseries.empty:  # 将第一个合约的数据放入左推 DataFrame 的第一列
            curve_structure_timeseries = current_data
            continue
        else:  # 合并新合约的数据
            curve_structure_timeseries = pd.merge(
                curve_structure_timeseries,
                current_data,
                on = 'date',
                how = 'outer'
            )

    def Left_Aligne(curve_structure_timeseries, max_len = 12):
        timeseries_values = curve_structure_timeseries.iloc[:, 0].values  # 时间列的 NumPy 数组
        data_values = curve_structure_timeseries.iloc[:, 1:].values  # 数据部分的 NumPy 数组

        # 一次性创建结果数组，包含日期列和数据列
        row_length = timeseries_values.shape[0]  # 使用时间行长度作为左推结果的长度
        left_aligned_data = np.empty((row_length, max_len + 1), dtype = object)
        left_aligned_data[:, 0] = timeseries_values  # 第一列是时间

        for i in range(row_length):
            row = data_values[i]
            row = row[ ~ pd.isna(row)]  # 非空掩码取得有值的行元素

            n_valid = min(len(row), max_len)
            if n_valid > 0:
                left_aligned_data[
                    i, 1 : 1+n_valid  # 列定位通过 +1 避开 date 列
                ] = row[: n_valid]  # 剩余部分保持为 None

        return pd.DataFrame(
            left_aligned_data, columns = ['date'] + [f'{i + 1}' for i in range(max_len)]
        )

    curve_structure_timeseries = Left_Aligne(curve_structure_timeseries, max_len = 12)
    curve_structure_timeseries = curve_structure_timeseries.set_index('date')
    curve_structure_timeseries = (
        curve_structure_timeseries
        .loc[main_continuous_minutely_timeseries['date']]
        .reset_index()
    )
    return curve_structure_timeseries

In [15]:
def correct_outliers(column, variety, curve_structure_timeseries):

    if column in ['volume', 'money'] and variety == 'RB':  # 螺纹钢 数据异常值修正
        return curve_structure_timeseries

    # 对指定时间的非日期列进行修正
    for col in curve_structure_timeseries.columns:
        if col == 'date':
            continue

        mask = curve_structure_timeseries['date'].isin([
            '2019-09-16 10:15:00', '2019-09-16 10:31:00', '2019-09-16 11:30:00'
        ])
        curve_structure_timeseries.loc[mask, col] /= 100
    return curve_structure_timeseries

In [16]:
# 临时实验数据品种文件夹路径 TED_variety_path
TED_variety_path = os.path.join(str(temporary_experimental_data_dir), str(variety))

# 时间范围文件夹路径 dataset_range_boundary_path
dataset_range_boundary_path = os.path.join(TED_variety_path, dataset_range_boundary_file_name)

for column in [
    'open', 'high', 'low', 'close',
    'volume', 'money',
    'open_interest'
]:
    curve_structure_timeseries = process_curve_structure_timeseries(column = column)  # 期限结构分钟数据时间序列文件路径 curve_structure_timeseries

    curve_structure_timeseries = correct_outliers(column, variety, curve_structure_timeseries)  # 螺纹钢 数据异常值修正
    curve_structure_timeseries_path = os.path.join(dataset_range_boundary_path, f'curve_structure_{column}_timeseries.csv')
    curve_structure_timeseries.to_csv(curve_structure_timeseries_path, index = False)

RB1101
Empty DataFrame
Columns: [date]
Index: []

RB1102
                     date  RB1101
4948  2011-01-17 14:59:00  4875.0
4949  2011-01-17 15:00:00  4875.0

RB1103
                     date  RB1101  RB1102
8548  2011-02-15 14:59:00     NaN  5090.0
8549  2011-02-15 15:00:00     NaN  5090.0

RB1104
                      date  RB1101  RB1102  RB1103
13048  2011-03-15 14:59:00     NaN     NaN  4500.0
13049  2011-03-15 15:00:00     NaN     NaN  4500.0

RB1105
                      date  RB1101  RB1102  RB1103  RB1104
17773  2011-04-15 14:59:00     NaN     NaN     NaN  4755.0
17774  2011-04-15 15:00:00     NaN     NaN     NaN  4755.0

RB1106
                      date  RB1101  RB1102  RB1103  RB1104  RB1105
22273  2011-05-16 14:59:00     NaN     NaN     NaN     NaN  5039.0
22274  2011-05-16 15:00:00     NaN     NaN     NaN     NaN  5039.0

RB1107
                      date  RB1101  RB1102  RB1103  RB1104  RB1105  RB1106
26998  2011-06-15 14:59:00     NaN     NaN     NaN     NaN     NaN  4